# Multi-Modal LLM using EdenAI for image reasoning

In this notebook, we show how to use MultiModal LLM class for image understanding/reasoning.


In [ ]:
%pip install llama-index-multi-modal-llms-edenai

## Load and initialize EdenAI 

In [1]:
import os


EDENAI_API_KEY = ""  # Your edenai API token here
os.environ["EDENAI_API_KEY"] = EDENAI_API_KEY


## Download Images and Load Images locally

In [4]:
import os
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO
from llama_index.core.multi_modal_llms.generic_utils import load_image_urls
from llama_index.core.schema import ImageDocument

if not os.path.exists("test_images"):
    os.makedirs("test_images")

image_urls = [
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRFU7U2h0umyF0P6E_yhTX45sGgPEQAbGaJ4g&s",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/481px-Cat03.jpg",
]

image_documents = [
    ImageDocument(image_url=url)
    for url in image_urls
]


## Provide various prompts to test different Multi Modal LLMs

In [5]:
from llama_index.multi_modal_llms.edenai import EdenaiMultiModal

prompts = [
    "what is shown in this image?",
    "what animal is in the picture?",
]


## Generate Image Reasoning from different LLMs with different prompts for different images

In [ ]:
res = []
llm_model = "google/gemini-1.5-flash"
for prompt_idx, prompt in enumerate(prompts):
    for image_idx, image_doc in enumerate(image_documents):
        try:
            multi_modal_llm = EdenaiMultiModal(
                model_name=llm_model,
                top_p=0.9,
                api_key=EDENAI_API_KEY,
            )

            mm_resp = multi_modal_llm.complete(
                prompt=prompt,
                image_documents=[image_doc],
            )
            print(mm_resp.raw)
        except Exception as e:
            print(
                f"Error with LLM model inference with prompt {prompt}, image {image_idx}, and MM model {llm_model}"
            )
            print("Inference Failed due to: ", e)
            continue
        res.append(
            {
                "model": llm_model,
                "prompt": prompt,
                "response": mm_resp,
            }
        )

## Display Sampled Responses from Multi-Modal LLMs 

### Using complete 

In [ ]:
from IPython.display import display
import pandas as pd

pd.options.display.max_colwidth = None
df = pd.DataFrame(res)
display(df[:5])

### Using async complete

In [11]:
resp = await multi_modal_llm.acomplete(
    prompt="tell me about this image",
    image_documents=[image_documents[0]],
)

In [ ]:
print(resp)

### Using chat 


In [ ]:
from llama_index.multi_modal_llms.edenai import EdenaiMultiModal
from llama_index.core.llms import (
    ChatMessage,
    ImageBlock,
    TextBlock,
    MessageRole,
)

llm_model = "openai/gpt-4o"
results = []
for prompt_idx, prompt in enumerate(prompts):
    for image_idx, image_doc in enumerate(image_documents):
        try:
            messages = [
                ChatMessage(
                    role=MessageRole.USER,
                    blocks=[
                        TextBlock(text=prompt),
                        ImageBlock(image=None, path=None,url=image_doc.image_url)  
                    ]
                )
            ]

            multi_modal_llm = EdenaiMultiModal(
                model_name=llm_model,
                top_p=0.9,
                api_key=EDENAI_API_KEY,
            )

            mm_resp = multi_modal_llm.chat(messages=messages)

        except Exception as e:
            print(
                f"Error with LLM model inference: prompt '{prompt}', image {image_idx}, model '{llm_model}'."
            )
            print("Inference Failed due to: ", e)
            continue

        results.append(
            {
                "model": llm_model,
                "prompt": prompt,
                "response": mm_resp,
            }
        )


In [ ]:
from IPython.display import display
import pandas as pd

pd.options.display.max_colwidth = None
df = pd.DataFrame(results)
display(df[:5])